In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from shapely.geometry import box

In [ ]:
import os
import sys
import cv2

# Compute absolute path to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), "."))
src_path = os.path.join(project_root, "src")

# Add src/ to sys.path
if src_path not in sys.path:
    print(f"Adding {src_path} to PYTHONPATH")
    sys.path.insert(0, src_path)

In [ ]:
from src.fsm import *

# 🏴‍☠️ Introduction

The following notebook presents our work on the **Pirate Robot Navigation Project** for the Basics of Mobile Robotics course.  
This notebook contains the conceptual description of each module used in the system, while the complete implementation is stored inside separate Python files for clarity.

Our robot, imagined as a pirate vessel sailing on a digital ocean, must autonomously travel from a starting island to a final treasure island.  
Throughout its journey, the robot faces **continents, reefs, cliffs, and forbidden shallow waters**, all of which are extracted from the camera feed and represented as polygonal obstacles.

Just like a real pirate navigating dangerous seas:

- it must avoid colliding with reefs,
- it must find narrow passages between continents,
- and it must deal with imperfect, noisy sensor information.

In order to reach the treasure efficiently and safely, the robot must compute a **global collision-free trajectory**, estimate its position robustly using filtering, and execute precise **local navigation**.

We structured the project as follows:

- **Vision** 
- **Kalman filter** 
- **Global path planning** 
- **Local avoidance**
- **Control action**

Each section focuses on the ideas, algorithms, and design choices rather than the raw code.


# 🗺️ Project Description

The objective of this project is to navigate our pirate-themed robot from a departure island to a remote treasure location while avoiding continents and shallow waters that appear on its map.

The robot must:

- compute a **global optimal path** free of collisions,
- react to **unplanned obstacles** during the journey,
- determine its position from an **overhead camera**,
- correct noisy measurements via a **Kalman filter**,
- and follow the computed path precisely using **local motion control**.

The environment is represented as a 2D map where dark shapes correspond to continents or reefs.  
The robot’s mission is to cross this environment safely, charting a valid sea route all the way to the treasure island.


# 1. Vision pipeline walkthrough

Step-by-step view of the vision chain. Quick pipeline: capture one frame -> color masks (HSV for robot/goal) -> Canny -> obstacle polygons -> px/mm conversions. Key functions:
- `find_available_camera`: picks the available webcam (USB/macOS priority) for live runs.
- `InitVisionWizard`: Tk wizard to tune HSV/Canny/polygon area and save into `vision_params.json`.
- `vision_params_manager`: load/save the thresholds.
- `ComputerVisionCore`: `detect_robot`, `detect_goal`, `init_canny`, `detect_obstacles` (Shapely polygons), and px/mm conversions.
- `Vision`: wraps the core and smooths detections (cache + alpha).

### Functions at a glance
- `detect_robot(frame, params_color)`: find red/green markers, return center/theta and an overlay mask.
- `detect_goal(frame, params_color)`: isolate the blue goal marker and its center.
- `init_canny(frame, sigma)`: auto-select Canny low/high thresholds from image median.
- `detect_obstacles(frame, params_color, params_canny, params_poly)`: contours -> convex polygons, excluding robot/goal areas.
- `Vision(params_color, params_canny, params_poly)`: wrapper that caches + smooths detections across frames.
- `find_available_camera()`: choose the best webcam index (env override, macOS/USB priority).
- `InitVisionWizard`: Tk UI to tune HSV/Canny/polygon area and save to `vision_params.json`.


### Initialize Computer Vision or Load Parameters and Capture a Fixed Frame

The system detects the robot using two colored circular markers: red and green.
An initialization wizard `InitVisionWizard.py` provides interactive sliders to adjust HSV thresholds (Hue min/max for each color, plus global Saturation/Value limits).
After tuning, all parameters are saved in vision_params.json, so calibration only needs to be done once.

A captured frame is converted to HSV, thresholded, and circular blobs are extracted.
From these blobs, the robot’s center, orientation, and heading vector are computed.

### Method used for robust detection

We extract red and green masks in HSV, detect circular contours, and retain only blobs with the correct size and high circularity.
Each red–green pair is evaluated:
- the two blobs must have similar radius,
- and be within a distance proportional to their size,
ensuring they truly belong to the same robot.

Every valid pair receives a score based on circularity, radius similarity, and distance.
The best-scoring pair is kept.

The robot’s center is the midpoint between the blobs, and its orientation is the direction from red to green (with a 90° correction).

All of this detection logic is implemented in `computer_vision.py`.

In [ ]:


img_path = Path.cwd() / "imgs" / "slider_init_vision.png"  
img = Image.open(img_path)
plt.imshow(img)
plt.axis("off")
plt.show()

### Canny edges init

init_canny provides sliders to tune the Canny thresholds. Before running Canny, we convert the frame to grayscale and apply a median blur to reduce noise while preserving sharp edges. We then extract edges with Canny, close small gaps with a morphological closing operation, and find external contours. Small contours below a configurable minimum area are discarded, and the remaining ones are converted into convex polygons.

```python
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
blur = cv2.medianBlur(gray, 5)
edges = cv2.Canny(blur, params_canny["low"], params_canny["high"])
edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE,cv2.getStructuringElement(cv2.MORPH_RECT, (5,5))) #closed shape
cnts = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

for c in cnts:
    area = cv2.contourArea(c)
        if area < params_poly["min_area"]:
            continue
        hull = cv2.convexHull(c) #to give convex shap 
        if len(hull) >= 3:  # at least a triangle
            poly = Polygon([(pt[0][0], pt[0][1]) for pt in hull])
            # Skip polygon if it intersects with robot bounding box
            if (robot_bbox is not None and poly.intersects(robot_bbox)) or (goal_bbox is not None and poly.intersects(goal_bbox)): 
                continue
            obstacles.append(poly)
```

In [ ]:
img_path = Path.cwd() / "imgs" / "canny.png"   # ajuste si l’image est ailleurs

img = Image.open(img_path)
plt.imshow(img)
plt.axis("off")
plt.show()

### Detected obstacle, goal and robot to give to path planning

From the processed image, we extract the robot pose \((x_r, y_r, \theta_r)\), the goal position \((x_g, y_g)\), and all detected obstacles as a set of points or discs.  
These elements are converted to world coordinates and packed into a structured message, which is then sent to the path

In [ ]:
img_path = Path.cwd() / "imgs" / "obstacle.png"   # ajuste si l’image est ailleurs

img = Image.open(img_path)
plt.imshow(img)
plt.axis("off")
plt.show()

# 2. State estimation

The purpose of state estimation is to reduce the uncertainty of the physical world to which our robot is subjected. During its treasure hunt, our Thymio robot evolves by detecting its environment and performing actions based on mathematical representations of that environment. Although this environment appears relatively straightforward, it is inherently unpredictable. This is partly because the robot can only perceive it through imperfect sensors that are difficult to model due to the complexity of their manufacturing processes. However, the environment is also unpredictable due to its variability over time.

Therefore, we need a model that can filter out this uncertainty, enabling our robot to perform its tasks with greater robustness. However, in order to select such a model, we must first define the state-space representation of our system.

### Treasure referentials definition

The following schematic illustrates the coordinate frames and reference points used to define the robot, goal, and environment in the treasure-hunting task.

<div style="text-align: center;">
    <img src="./imgs/map_drawing.jpeg" alt="Treasure map" width="800"/>
</div>

### State-space representation

#### State variable definition

As we have seen, the state variable is the smallest possible subset of variables required to describe the evolution of the robot over time. It means that it should contain only the important quantities that are affecting the robot state and no redundant informations. For this project, we'll define the state of the Thymio as follow:

- Thymio is a differential dirve robot (DDR) with 3 DOFs, so three variables are required to describe its pose: $x$, $y$ and $\theta$

- Given that the robot is moving at a constant linear and/or angular velocity, its pose at the next time step can be computed directly from its previous pose and the measured speeds of its motors. Therefore, we can expressed its motion model in its referential frame as follow:

```math
\begin{equation*}
    \begin{align*}
        x_{k} &= x_{k-1} + \Delta T v \\
        \theta_{k} &= \theta_{k-1} + \Delta T \omega
    \end{align*}
\end{equation*}
```

- As explained in [1-Components of a Mobile Robot, slide 11](https://moodle.epfl.ch/pluginfile.php/2703472/mod_resource/content/13/Slides%2001%20-%20Components%20of%20a%20Mobile%20Robot.pdf), the forward and angular speeds are related to the motor speeds with the following equations:

```math
\begin{equation*}
    \begin{align*}
        v &= \frac{r\dot{\phi}_r}{2} + \frac{r\dot{\phi}_l}{2} \\
        \omega &= \frac{r\dot{\phi}_r}{d}-\frac{r\dot{\phi}_l}{d}
    \end{align*}
\end{equation*}
```

Where $\dot{\phi}_i$ is the motor speed, $r$ the wheel radius and $d$ the distance between both wheels.

- Although the motor speeds $\dot{\phi}_r$ and $\dot{\phi}_l$ appear in the motion model, they are not quantities that the filter must estimate: they are directly provided by the robot’s speed sensors (odometry) or motor commands. Once these speeds are used to predict the position, they are no longer needed as part of the state. Including them as states would add dimensions to the covariance matrix and increase computational complexity without providing additional useful information.

Hence, we have decided to use the following state definition: $\vec{x}(t) =
\begin{pmatrix}
  x\\ 
  y\\
  \theta
\end{pmatrix}$.

#### State-space equations

Expressing the Thymio’s state in a fixed reference frame relative to the map leads to a nonlinear system, because the robot’s velocity must be projected onto the reference axes using trigonometric (sine and cosine) terms. To map the velocities from the robot's referential frame to the global reference frame of the map, we use rotational matrices as we have seen in [1-Components of a Mobile Robot, slide 12](https://moodle.epfl.ch/pluginfile.php/2703472/mod_resource/content/13/Slides%2001%20-%20Components%20of%20a%20Mobile%20Robot.pdf):

- To align both x axis, we perform a rotation around the z axis of the global frame, using:

```math
\begin{equation*}
    R_z(-\theta) = R_z(\theta)^{-1} =
    \begin{pmatrix}
        cos(\theta) & sin(\theta) & 0 \\
        -sin(\theta) & cos(\theta) & 0 \\
        0 & 0 & 1
    \end{pmatrix}
\end{equation*}
```

- Then to reverse the z axis, we perform a rotation around the x axis of the global frame, using:

```math
\begin{equation*}
    R_x(\pi) =
    \begin{pmatrix}
        1 & 0 & 0 \\
        0 & -1 & 0 \\
        0 & 0 & -1
    \end{pmatrix}
\end{equation*}
```

- Both rotations are illustrated bellow:

<div style="text-align: center;">
    <img src="./imgs/referential_rotation.jpeg" alt="Treasure map" width="800"/>
</div>

Therefore, the velocities of the robot in the global referential frame are:

```math
\begin{equation*}
    \begin{pmatrix}
        \dot{x} \\
        \dot{y} \\
        \dot{\theta}
    \end{pmatrix} = R_x(\pi)R_z(-\theta)
    \begin{pmatrix}
        v \\
        0 \\
        \omega
    \end{pmatrix} = 
    \begin{pmatrix}
        cos(\theta) & sin(\theta) & 0 \\
        sin(\theta) & -cos(\theta) & 0 \\
        0 & 0 & -1
    \end{pmatrix}
    \begin{pmatrix}
        v \\
        0 \\
        \omega
    \end{pmatrix} =
    \begin{pmatrix}
        vcos(\theta) \\
        vsin(\theta) \\
        -\omega
    \end{pmatrix}
\end{equation*}
```

We can then use these, along with the previous relationship between motor speeds, to define the discrete state-space equations:

```math
\begin{equation*}
    \begin{align*}
        \vec{x}_{k} &= 
            \begin{pmatrix}
                x_{k}\\ 
                y_{k}\\
                \theta_{k}
            \end{pmatrix} 
            = \boldsymbol{g}(\vec{x}_{k-1}, \vec{u}_k) = 
            \begin{pmatrix}
                x_{k-1} + dt \cdot (\frac{r\dot{\phi}_r}{2} + \frac{r\dot{\phi}_l}{2}) \cdot cos(\theta_{k-1})\\ 
                y_{k-1} + dt \cdot (\frac{r\dot{\phi}_r}{2} + \frac{r\dot{\phi}_l}{2}) \cdot sin(\theta_{k-1})\\
                \theta_{k-1} - dt \cdot (\frac{r\dot{\phi}_r}{d} - \frac{r\dot{\phi}_l}{d})
            \end{pmatrix} =
            \begin{pmatrix}
                x_{k-1} + dt \cdot \frac{\lambda}{2} (u_{r,k} + u_{l,k}) \cdot cos(\theta_{k-1})\\ 
                y_{k-1} + dt \cdot \frac{\lambda}{2} (u_{r,k} + u_{l,k}) \cdot sin(\theta_{k-1})\\
                \theta_{k-1} - dt \cdot \frac{\lambda}{d} (u_{r,k} - u_{l,k})
            \end{pmatrix} \quad \text{with} \quad \vec{u}_k =
                \begin{pmatrix}
                    u_{r,k} \\
                    u_{l,k}
                \end{pmatrix} \\
        \vec{z}_k &= 
            \begin{pmatrix}
                x_{k,measured} \\
                y_{k,measured} \\
                \theta_{k,measured}
            \end{pmatrix} = \boldsymbol{h}(\vec{x}_{k}) =
            \begin{pmatrix}
                x_k \\
                y_k \\
                \theta_k
            \end{pmatrix}
    \end{align*}
\end{equation*}
```

Where $dt$ is the sampling period, $\vec{u}_{k}$ is the control input provided by the odometry (i.e., the measured wheel revolutions), and $\lambda$ is a conversion factor that converts motor commands into a linear speed expressed in $mm/s$. 

Notice that the measurement model simply involves directly observing the robot's pose using the camera.

Now that we have defined the state-space representation of our system, we need to estimate the type of noise that will affect it and select an appropriate uncertainty filter algorithm from those discussed in class.

### State-space parameters

Converting the previously defined state-space representation in python requires to fix numerical values for the $dt$ and $\lambda$, which only depend on the Thymio robot. 

#### Sampling time $dt$

In order to define the *Extended Kalman Filter* sampling time $dt$, we looked into the effective sampling rates of the different sensors we are going to use in this project and we obtain the following list:
- Motors variables update - $100 \ Hz$

- Camera frame update - $30 \ Hz$

- Proximity sensors update - $10 \ Hz$

However, although the motor control loop is specified in the datasheet as running at 100 Hz, we discovered that the speed sensing subsystem does not provide new measurements at this rate. By logging sensor values while attempting to read the motor speed at 100 Hz, we observed that the returned values remained constant for blocks of approximately ten consecutive samples before changing. In other words, even though the motors themselves can be driven at 100 Hz, the measured odometry information is effectively limited to 10 Hz. 

This experiment confirmed our decision to choose $dt = 0.1 \ s$, which is consistent with the recommended choice written in Exercise 8.

Hence we have decided to define:

In [ ]:
dt = 0.1 # Time step (s)

In [ ]:
from tdmclient import ClientAsync, aw
client = ClientAsync()
node = aw(client.wait_for_node())
aw(node.lock())

In [ ]:
from src.utils.speed_variance_calculator import SpeedVarianceCalculator

In [ ]:
aquire_data = False

if aquire_data:
    # Experiment parameters
    tp8_exp = SpeedVarianceCalculator(0.1, 30, 50, 50)
    tp8_exp.plot_ground_sensor_detection(400)
    lambda_ = tp8_exp.convert_thymio_speed_to_mm_s()
    
else:
    # Hardcoded data from experiment
    data = [{'ground': [840, 916], 'sensor': [840, 916], 'left_speed': 0, 'right_speed': 0}, {'ground': [832, 912], 'sensor': [832, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [831, 909], 'sensor': [831, 909], 'left_speed': 0, 'right_speed': 0}, {'ground': [831, 911], 'sensor': [831, 911], 'left_speed': 0, 'right_speed': 0}, {'ground': [829, 908], 'sensor': [829, 908], 'left_speed': 0, 'right_speed': 0}, {'ground': [831, 918], 'sensor': [831, 918], 'left_speed': 0, 'right_speed': 0}, {'ground': [832, 909], 'sensor': [832, 909], 'left_speed': 0, 'right_speed': 0}, {'ground': [831, 910], 'sensor': [831, 910], 'left_speed': 0, 'right_speed': 0}, {'ground': [830, 911], 'sensor': [830, 911], 'left_speed': 0, 'right_speed': 0}, {'ground': [832, 912], 'sensor': [832, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [832, 910], 'sensor': [832, 910], 'left_speed': 0, 'right_speed': 0}, {'ground': [830, 912], 'sensor': [830, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [836, 912], 'sensor': [836, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [840, 912], 'sensor': [840, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [836, 912], 'sensor': [836, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [832, 911], 'sensor': [832, 911], 'left_speed': 0, 'right_speed': 0}, {'ground': [831, 919], 'sensor': [831, 919], 'left_speed': 0, 'right_speed': 0}, {'ground': [833, 912], 'sensor': [833, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [834, 919], 'sensor': [834, 919], 'left_speed': 0, 'right_speed': 0}, {'ground': [835, 912], 'sensor': [835, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [840, 912], 'sensor': [840, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [833, 912], 'sensor': [833, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [834, 916], 'sensor': [834, 916], 'left_speed': 0, 'right_speed': 0}, {'ground': [839, 910], 'sensor': [839, 910], 'left_speed': 0, 'right_speed': 0}, {'ground': [832, 911], 'sensor': [832, 911], 'left_speed': 0, 'right_speed': 0}, {'ground': [832, 910], 'sensor': [832, 910], 'left_speed': 0, 'right_speed': 0}, {'ground': [835, 911], 'sensor': [835, 911], 'left_speed': 0, 'right_speed': 0}, {'ground': [839, 916], 'sensor': [839, 916], 'left_speed': 0, 'right_speed': 0}, {'ground': [836, 912], 'sensor': [836, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [833, 910], 'sensor': [833, 910], 'left_speed': 0, 'right_speed': 0}, {'ground': [840, 918], 'sensor': [840, 918], 'left_speed': 0, 'right_speed': 0}, {'ground': [832, 909], 'sensor': [832, 909], 'left_speed': 0, 'right_speed': 0}, {'ground': [833, 912], 'sensor': [833, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [840, 912], 'sensor': [840, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [831, 912], 'sensor': [831, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [832, 911], 'sensor': [832, 911], 'left_speed': 0, 'right_speed': 0}, {'ground': [832, 911], 'sensor': [832, 911], 'left_speed': 0, 'right_speed': 0}, {'ground': [832, 910], 'sensor': [832, 910], 'left_speed': 0, 'right_speed': 0}, {'ground': [834, 916], 'sensor': [834, 916], 'left_speed': 0, 'right_speed': 0}, {'ground': [832, 911], 'sensor': [832, 911], 'left_speed': 0, 'right_speed': 0}, {'ground': [833, 912], 'sensor': [833, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [836, 912], 'sensor': [836, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [831, 914], 'sensor': [831, 914], 'left_speed': 0, 'right_speed': 0}, {'ground': [835, 911], 'sensor': [835, 911], 'left_speed': 0, 'right_speed': 0}, {'ground': [831, 911], 'sensor': [831, 911], 'left_speed': 0, 'right_speed': 0}, {'ground': [832, 911], 'sensor': [832, 911], 'left_speed': 0, 'right_speed': 0}, {'ground': [832, 912], 'sensor': [832, 912], 'left_speed': 0, 'right_speed': 0}, {'ground': [839, 909], 'sensor': [839, 909], 'left_speed': 0, 'right_speed': 0}, {'ground': [832, 912], 'sensor': [832, 912], 'left_speed': 27, 'right_speed': 25}, {'ground': [838, 912], 'sensor': [838, 912], 'left_speed': 37, 'right_speed': 37}, {'ground': [844, 923], 'sensor': [844, 923], 'left_speed': 44, 'right_speed': 45}, {'ground': [847, 918], 'sensor': [847, 918], 'left_speed': 46, 'right_speed': 41}, {'ground': [847, 918], 'sensor': [847, 918], 'left_speed': 46, 'right_speed': 41}, {'ground': [854, 918], 'sensor': [854, 918], 'left_speed': 52, 'right_speed': 47}, {'ground': [854, 918], 'sensor': [854, 918], 'left_speed': 51, 'right_speed': 47}, {'ground': [849, 920], 'sensor': [849, 920], 'left_speed': 51, 'right_speed': 47}, {'ground': [834, 919], 'sensor': [834, 919], 'left_speed': 49, 'right_speed': 50}, {'ground': [834, 919], 'sensor': [834, 919], 'left_speed': 49, 'right_speed': 50}, {'ground': [819, 896], 'sensor': [819, 896], 'left_speed': 49, 'right_speed': 52}, {'ground': [721, 738], 'sensor': [721, 738], 'left_speed': 50, 'right_speed': 53}, {'ground': [544, 490], 'sensor': [544, 490], 'left_speed': 52, 'right_speed': 54}, {'ground': [356, 262], 'sensor': [356, 262], 'left_speed': 52, 'right_speed': 54}, {'ground': [176, 128], 'sensor': [176, 128], 'left_speed': 51, 'right_speed': 54}, {'ground': [113, 115], 'sensor': [113, 115], 'left_speed': 51, 'right_speed': 53}, {'ground': [108, 113], 'sensor': [108, 113], 'left_speed': 50, 'right_speed': 52}, {'ground': [101, 111], 'sensor': [101, 111], 'left_speed': 50, 'right_speed': 51}, {'ground': [102, 109], 'sensor': [102, 109], 'left_speed': 49, 'right_speed': 45}, {'ground': [100, 110], 'sensor': [100, 110], 'left_speed': 48, 'right_speed': 50}, {'ground': [101, 119], 'sensor': [101, 119], 'left_speed': 50, 'right_speed': 51}, {'ground': [108, 113], 'sensor': [108, 113], 'left_speed': 50, 'right_speed': 49}, {'ground': [107, 113], 'sensor': [107, 113], 'left_speed': 50, 'right_speed': 49}, {'ground': [107, 116], 'sensor': [107, 116], 'left_speed': 50, 'right_speed': 45}, {'ground': [103, 122], 'sensor': [103, 122], 'left_speed': 50, 'right_speed': 44}, {'ground': [103, 124], 'sensor': [103, 124], 'left_speed': 47, 'right_speed': 44}, {'ground': [103, 119], 'sensor': [103, 119], 'left_speed': 47, 'right_speed': 45}, {'ground': [102, 124], 'sensor': [102, 124], 'left_speed': 48, 'right_speed': 45}, {'ground': [100, 121], 'sensor': [100, 121], 'left_speed': 49, 'right_speed': 45}, {'ground': [102, 112], 'sensor': [102, 112], 'left_speed': 48, 'right_speed': 45}, {'ground': [102, 111], 'sensor': [102, 111], 'left_speed': 50, 'right_speed': 46}, {'ground': [104, 113], 'sensor': [104, 113], 'left_speed': 48, 'right_speed': 46}, {'ground': [105, 112], 'sensor': [105, 112], 'left_speed': 48, 'right_speed': 47}, {'ground': [107, 121], 'sensor': [107, 121], 'left_speed': 50, 'right_speed': 48}, {'ground': [140, 209], 'sensor': [140, 209], 'left_speed': 50, 'right_speed': 50}, {'ground': [256, 382], 'sensor': [256, 382], 'left_speed': 52, 'right_speed': 56}, {'ground': [375, 486], 'sensor': [375, 486], 'left_speed': 54, 'right_speed': 56}, {'ground': [422, 462], 'sensor': [422, 462], 'left_speed': 54, 'right_speed': 59}, {'ground': [380, 325], 'sensor': [380, 325], 'left_speed': 54, 'right_speed': 55}, {'ground': [122, 116], 'sensor': [122, 116], 'left_speed': 54, 'right_speed': 53}, {'ground': [100, 112], 'sensor': [100, 112], 'left_speed': 53, 'right_speed': 52}, {'ground': [100, 110], 'sensor': [100, 110], 'left_speed': 51, 'right_speed': 53}, {'ground': [99, 113], 'sensor': [99, 113], 'left_speed': 48, 'right_speed': 51}, {'ground': [98, 116], 'sensor': [98, 116], 'left_speed': 47, 'right_speed': 48}, {'ground': [97, 117], 'sensor': [97, 117], 'left_speed': 48, 'right_speed': 48}, {'ground': [102, 108], 'sensor': [102, 108], 'left_speed': 48, 'right_speed': 45}, {'ground': [101, 108], 'sensor': [101, 108], 'left_speed': 47, 'right_speed': 43}, {'ground': [101, 105], 'sensor': [101, 105], 'left_speed': 46, 'right_speed': 43}, {'ground': [103, 109], 'sensor': [103, 109], 'left_speed': 46, 'right_speed': 43}, {'ground': [99, 107], 'sensor': [99, 107], 'left_speed': 46, 'right_speed': 45}, {'ground': [102, 108], 'sensor': [102, 108], 'left_speed': 47, 'right_speed': 46}, {'ground': [105, 107], 'sensor': [105, 107], 'left_speed': 48, 'right_speed': 49}, {'ground': [98, 106], 'sensor': [98, 106], 'left_speed': 49, 'right_speed': 48}, {'ground': [93, 101], 'sensor': [93, 101], 'left_speed': 48, 'right_speed': 49}, {'ground': [101, 106], 'sensor': [101, 106], 'left_speed': 52, 'right_speed': 49}, {'ground': [101, 107], 'sensor': [101, 107], 'left_speed': 52, 'right_speed': 51}, {'ground': [99, 111], 'sensor': [99, 111], 'left_speed': 53, 'right_speed': 52}, {'ground': [104, 110], 'sensor': [104, 110], 'left_speed': 52, 'right_speed': 55}, {'ground': [114, 161], 'sensor': [114, 161], 'left_speed': 53, 'right_speed': 56}, {'ground': [170, 295], 'sensor': [170, 295], 'left_speed': 54, 'right_speed': 55}, {'ground': [319, 468], 'sensor': [319, 468], 'left_speed': 53, 'right_speed': 54}, {'ground': [446, 570], 'sensor': [446, 570], 'left_speed': 51, 'right_speed': 51}, {'ground': [528, 592], 'sensor': [528, 592], 'left_speed': 50, 'right_speed': 49}, {'ground': [511, 496], 'sensor': [511, 496], 'left_speed': 49, 'right_speed': 48}, {'ground': [424, 420], 'sensor': [424, 420], 'left_speed': 49, 'right_speed': 45}, {'ground': [383, 404], 'sensor': [383, 404], 'left_speed': 47, 'right_speed': 44}, {'ground': [373, 406], 'sensor': [373, 406], 'left_speed': 46, 'right_speed': 42}, {'ground': [375, 407], 'sensor': [375, 407], 'left_speed': 46, 'right_speed': 43}, {'ground': [367, 408], 'sensor': [367, 408], 'left_speed': 46, 'right_speed': 43}, {'ground': [366, 411], 'sensor': [366, 411], 'left_speed': 47, 'right_speed': 44}, {'ground': [363, 412], 'sensor': [363, 412], 'left_speed': 48, 'right_speed': 48}, {'ground': [365, 412], 'sensor': [365, 412], 'left_speed': 49, 'right_speed': 50}, {'ground': [369, 415], 'sensor': [369, 415], 'left_speed': 50, 'right_speed': 51}, {'ground': [371, 415], 'sensor': [371, 415], 'left_speed': 52, 'right_speed': 52}, {'ground': [368, 416], 'sensor': [368, 416], 'left_speed': 52, 'right_speed': 56}, {'ground': [370, 414], 'sensor': [370, 414], 'left_speed': 53, 'right_speed': 57}, {'ground': [371, 411], 'sensor': [371, 411], 'left_speed': 56, 'right_speed': 56}, {'ground': [367, 410], 'sensor': [367, 410], 'left_speed': 56, 'right_speed': 57}, {'ground': [363, 414], 'sensor': [363, 414], 'left_speed': 51, 'right_speed': 51}, {'ground': [368, 411], 'sensor': [368, 411], 'left_speed': 51, 'right_speed': 51}, {'ground': [373, 411], 'sensor': [373, 411], 'left_speed': 49, 'right_speed': 49}, {'ground': [375, 416], 'sensor': [375, 416], 'left_speed': 48, 'right_speed': 47}, {'ground': [380, 426], 'sensor': [380, 426], 'left_speed': 48, 'right_speed': 45}, {'ground': [389, 473], 'sensor': [389, 473], 'left_speed': 46, 'right_speed': 44}, {'ground': [454, 578], 'sensor': [454, 578], 'left_speed': 45, 'right_speed': 43}, {'ground': [555, 656], 'sensor': [555, 656], 'left_speed': 46, 'right_speed': 43}, {'ground': [571, 550], 'sensor': [571, 550], 'left_speed': 44, 'right_speed': 43}, {'ground': [472, 449], 'sensor': [472, 449], 'left_speed': 45, 'right_speed': 43}, {'ground': [403, 420], 'sensor': [403, 420], 'left_speed': 44, 'right_speed': 42}, {'ground': [386, 418], 'sensor': [386, 418], 'left_speed': 46, 'right_speed': 48}, {'ground': [376, 416], 'sensor': [376, 416], 'left_speed': 47, 'right_speed': 48}, {'ground': [372, 413], 'sensor': [372, 413], 'left_speed': 48, 'right_speed': 48}, {'ground': [372, 413], 'sensor': [372, 413], 'left_speed': 49, 'right_speed': 48}, {'ground': [377, 416], 'sensor': [377, 416], 'left_speed': 49, 'right_speed': 50}, {'ground': [376, 417], 'sensor': [376, 417], 'left_speed': 52, 'right_speed': 50}, {'ground': [380, 416], 'sensor': [380, 416], 'left_speed': 51, 'right_speed': 52}, {'ground': [383, 422], 'sensor': [383, 422], 'left_speed': 51, 'right_speed': 54}, {'ground': [388, 418], 'sensor': [388, 418], 'left_speed': 53, 'right_speed': 57}, {'ground': [388, 421], 'sensor': [388, 421], 'left_speed': 53, 'right_speed': 56}, {'ground': [388, 427], 'sensor': [388, 427], 'left_speed': 52, 'right_speed': 57}, {'ground': [389, 428], 'sensor': [389, 428], 'left_speed': 53, 'right_speed': 56}, {'ground': [389, 427], 'sensor': [389, 427], 'left_speed': 53, 'right_speed': 57}, {'ground': [387, 421], 'sensor': [387, 421], 'left_speed': 53, 'right_speed': 57}, {'ground': [382, 413], 'sensor': [382, 413], 'left_speed': 54, 'right_speed': 57}, {'ground': [369, 405], 'sensor': [369, 405], 'left_speed': 52, 'right_speed': 50}, {'ground': [360, 399], 'sensor': [360, 399], 'left_speed': 51, 'right_speed': 46}, {'ground': [349, 398], 'sensor': [349, 398], 'left_speed': 48, 'right_speed': 44}, {'ground': [359, 452], 'sensor': [359, 452], 'left_speed': 47, 'right_speed': 43}, {'ground': [422, 548], 'sensor': [422, 548], 'left_speed': 44, 'right_speed': 43}, {'ground': [422, 548], 'sensor': [422, 548], 'left_speed': 44, 'right_speed': 43}, {'ground': [516, 587], 'sensor': [516, 587], 'left_speed': 45, 'right_speed': 44}, {'ground': [511, 475], 'sensor': [511, 475], 'left_speed': 43, 'right_speed': 43}, {'ground': [389, 289], 'sensor': [389, 289], 'left_speed': 44, 'right_speed': 43}, {'ground': [224, 150], 'sensor': [224, 150], 'left_speed': 45, 'right_speed': 44}, {'ground': [131, 120], 'sensor': [131, 120], 'left_speed': 47, 'right_speed': 44}, {'ground': [112, 115], 'sensor': [112, 115], 'left_speed': 49, 'right_speed': 46}, {'ground': [97, 108], 'sensor': [97, 108], 'left_speed': 49, 'right_speed': 49}, {'ground': [97, 109], 'sensor': [97, 109], 'left_speed': 52, 'right_speed': 52}, {'ground': [95, 112], 'sensor': [95, 112], 'left_speed': 51, 'right_speed': 54}, {'ground': [96, 116], 'sensor': [96, 116], 'left_speed': 54, 'right_speed': 56}, {'ground': [97, 121], 'sensor': [97, 121], 'left_speed': 53, 'right_speed': 56}, {'ground': [99, 126], 'sensor': [99, 126], 'left_speed': 54, 'right_speed': 56}, {'ground': [102, 123], 'sensor': [102, 123], 'left_speed': 53, 'right_speed': 53}, {'ground': [103, 122], 'sensor': [103, 122], 'left_speed': 51, 'right_speed': 52}, {'ground': [103, 122], 'sensor': [103, 122], 'left_speed': 49, 'right_speed': 49}, {'ground': [103, 128], 'sensor': [103, 128], 'left_speed': 48, 'right_speed': 47}, {'ground': [106, 128], 'sensor': [106, 128], 'left_speed': 49, 'right_speed': 45}, {'ground': [103, 126], 'sensor': [103, 126], 'left_speed': 46, 'right_speed': 44}, {'ground': [99, 127], 'sensor': [99, 127], 'left_speed': 46, 'right_speed': 43}, {'ground': [97, 118], 'sensor': [97, 118], 'left_speed': 45, 'right_speed': 43}, {'ground': [100, 120], 'sensor': [100, 120], 'left_speed': 48, 'right_speed': 47}, {'ground': [95, 119], 'sensor': [95, 119], 'left_speed': 50, 'right_speed': 51}, {'ground': [90, 120], 'sensor': [90, 120], 'left_speed': 51, 'right_speed': 52}, {'ground': [94, 134], 'sensor': [94, 134], 'left_speed': 54, 'right_speed': 53}, {'ground': [123, 244], 'sensor': [123, 244], 'left_speed': 54, 'right_speed': 57}, {'ground': [267, 425], 'sensor': [267, 425], 'left_speed': 55, 'right_speed': 56}, {'ground': [428, 503], 'sensor': [428, 503], 'left_speed': 53, 'right_speed': 56}, {'ground': [469, 455], 'sensor': [469, 455], 'left_speed': 52, 'right_speed': 57}, {'ground': [384, 286], 'sensor': [384, 286], 'left_speed': 51, 'right_speed': 55}, {'ground': [228, 145], 'sensor': [228, 145], 'left_speed': 51, 'right_speed': 53}, {'ground': [126, 114], 'sensor': [126, 114], 'left_speed': 48, 'right_speed': 49}, {'ground': [96, 114], 'sensor': [96, 114], 'left_speed': 48, 'right_speed': 47}, {'ground': [104, 108], 'sensor': [104, 108], 'left_speed': 48, 'right_speed': 46}, {'ground': [108, 111], 'sensor': [108, 111], 'left_speed': 45, 'right_speed': 45}, {'ground': [104, 117], 'sensor': [104, 117], 'left_speed': 44, 'right_speed': 45}, {'ground': [106, 115], 'sensor': [106, 115], 'left_speed': 45, 'right_speed': 42}, {'ground': [110, 126], 'sensor': [110, 126], 'left_speed': 46, 'right_speed': 42}, {'ground': [107, 118], 'sensor': [107, 118], 'left_speed': 45, 'right_speed': 43}, {'ground': [103, 112], 'sensor': [103, 112], 'left_speed': 45, 'right_speed': 45}, {'ground': [107, 111], 'sensor': [107, 111], 'left_speed': 46, 'right_speed': 49}, {'ground': [104, 111], 'sensor': [104, 111], 'left_speed': 48, 'right_speed': 50}, {'ground': [101, 112], 'sensor': [101, 112], 'left_speed': 50, 'right_speed': 52}, {'ground': [99, 114], 'sensor': [99, 114], 'left_speed': 51, 'right_speed': 50}, {'ground': [98, 111], 'sensor': [98, 111], 'left_speed': 53, 'right_speed': 57}, {'ground': [99, 109], 'sensor': [99, 109], 'left_speed': 54, 'right_speed': 57}, {'ground': [97, 106], 'sensor': [97, 106], 'left_speed': 53, 'right_speed': 56}, {'ground': [98, 113], 'sensor': [98, 113], 'left_speed': 53, 'right_speed': 57}, {'ground': [107, 114], 'sensor': [107, 114], 'left_speed': 53, 'right_speed': 57}, {'ground': [111, 177], 'sensor': [111, 177], 'left_speed': 52, 'right_speed': 57}, {'ground': [185, 324], 'sensor': [185, 324], 'left_speed': 53, 'right_speed': 56}, {'ground': [342, 459], 'sensor': [342, 459], 'left_speed': 52, 'right_speed': 53}, {'ground': [430, 470], 'sensor': [430, 470], 'left_speed': 49, 'right_speed': 53}, {'ground': [432, 368], 'sensor': [432, 368], 'left_speed': 48, 'right_speed': 52}, {'ground': [313, 193], 'sensor': [313, 193], 'left_speed': 49, 'right_speed': 48}, {'ground': [156, 112], 'sensor': [156, 112], 'left_speed': 48, 'right_speed': 45}, {'ground': [105, 109], 'sensor': [105, 109], 'left_speed': 48, 'right_speed': 45}, {'ground': [101, 112], 'sensor': [101, 112], 'left_speed': 48, 'right_speed': 42}, {'ground': [102, 116], 'sensor': [102, 116], 'left_speed': 47, 'right_speed': 43}, {'ground': [98, 114], 'sensor': [98, 114], 'left_speed': 47, 'right_speed': 43}, {'ground': [98, 116], 'sensor': [98, 116], 'left_speed': 47, 'right_speed': 43}, {'ground': [100, 121], 'sensor': [100, 121], 'left_speed': 47, 'right_speed': 44}, {'ground': [102, 121], 'sensor': [102, 121], 'left_speed': 47, 'right_speed': 47}, {'ground': [100, 115], 'sensor': [100, 115], 'left_speed': 50, 'right_speed': 51}, {'ground': [99, 114], 'sensor': [99, 114], 'left_speed': 53, 'right_speed': 53}, {'ground': [101, 113], 'sensor': [101, 113], 'left_speed': 54, 'right_speed': 53}, {'ground': [95, 112], 'sensor': [95, 112], 'left_speed': 55, 'right_speed': 53}, {'ground': [100, 112], 'sensor': [100, 112], 'left_speed': 53, 'right_speed': 52}, {'ground': [100, 109], 'sensor': [100, 109], 'left_speed': 52, 'right_speed': 52}, {'ground': [98, 115], 'sensor': [98, 115], 'left_speed': 51, 'right_speed': 51}, {'ground': [97, 116], 'sensor': [97, 116], 'left_speed': 52, 'right_speed': 51}, {'ground': [98, 106], 'sensor': [98, 106], 'left_speed': 51, 'right_speed': 55}, {'ground': [92, 101], 'sensor': [92, 101], 'left_speed': 54, 'right_speed': 58}, {'ground': [90, 111], 'sensor': [90, 111], 'left_speed': 53, 'right_speed': 56}, {'ground': [93, 145], 'sensor': [93, 145], 'left_speed': 52, 'right_speed': 56}, {'ground': [125, 251], 'sensor': [125, 251], 'left_speed': 50, 'right_speed': 49}, {'ground': [249, 436], 'sensor': [249, 436], 'left_speed': 49, 'right_speed': 45}, {'ground': [411, 544], 'sensor': [411, 544], 'left_speed': 50, 'right_speed': 45}, {'ground': [516, 581], 'sensor': [516, 581], 'left_speed': 48, 'right_speed': 45}, {'ground': [527, 497], 'sensor': [527, 497], 'left_speed': 49, 'right_speed': 44}, {'ground': [441, 412], 'sensor': [441, 412], 'left_speed': 47, 'right_speed': 41}, {'ground': [361, 389], 'sensor': [361, 389], 'left_speed': 47, 'right_speed': 44}, {'ground': [358, 391], 'sensor': [358, 391], 'left_speed': 48, 'right_speed': 43}, {'ground': [353, 392], 'sensor': [353, 392], 'left_speed': 49, 'right_speed': 43}, {'ground': [352, 388], 'sensor': [352, 388], 'left_speed': 48, 'right_speed': 45}, {'ground': [358, 385], 'sensor': [358, 385], 'left_speed': 48, 'right_speed': 47}, {'ground': [356, 384], 'sensor': [356, 384], 'left_speed': 49, 'right_speed': 48}, {'ground': [354, 386], 'sensor': [354, 386], 'left_speed': 49, 'right_speed': 51}, {'ground': [350, 388], 'sensor': [350, 388], 'left_speed': 50, 'right_speed': 53}, {'ground': [351, 386], 'sensor': [351, 386], 'left_speed': 50, 'right_speed': 55}, {'ground': [351, 388], 'sensor': [351, 388], 'left_speed': 52, 'right_speed': 53}, {'ground': [352, 391], 'sensor': [352, 391], 'left_speed': 52, 'right_speed': 50}, {'ground': [352, 391], 'sensor': [352, 391], 'left_speed': 52, 'right_speed': 50}, {'ground': [348, 391], 'sensor': [348, 391], 'left_speed': 49, 'right_speed': 43}, {'ground': [351, 391], 'sensor': [351, 391], 'left_speed': 45, 'right_speed': 43}, {'ground': [352, 391], 'sensor': [352, 391], 'left_speed': 45, 'right_speed': 43}, {'ground': [358, 398], 'sensor': [358, 398], 'left_speed': 45, 'right_speed': 44}, {'ground': [363, 400], 'sensor': [363, 400], 'left_speed': 48, 'right_speed': 48}, {'ground': [367, 407], 'sensor': [367, 407], 'left_speed': 49, 'right_speed': 52}, {'ground': [373, 442], 'sensor': [373, 442], 'left_speed': 52, 'right_speed': 52}, {'ground': [412, 531], 'sensor': [412, 531], 'left_speed': 52, 'right_speed': 57}, {'ground': [511, 675], 'sensor': [511, 675], 'left_speed': 53, 'right_speed': 57}, {'ground': [630, 804], 'sensor': [630, 804], 'left_speed': 54, 'right_speed': 59}, {'ground': [743, 886], 'sensor': [743, 886], 'left_speed': 52, 'right_speed': 57}, {'ground': [743, 886], 'sensor': [743, 886], 'left_speed': 52, 'right_speed': 57}, {'ground': [808, 898], 'sensor': [808, 898], 'left_speed': 53, 'right_speed': 52}, {'ground': [824, 896], 'sensor': [824, 896], 'left_speed': 51, 'right_speed': 50}, {'ground': [825, 900], 'sensor': [825, 900], 'left_speed': 48, 'right_speed': 46}, {'ground': [830, 898], 'sensor': [830, 898], 'left_speed': 44, 'right_speed': 45}, {'ground': [823, 892], 'sensor': [823, 892], 'left_speed': 46, 'right_speed': 43}, {'ground': [808, 848], 'sensor': [808, 848], 'left_speed': 45, 'right_speed': 42}, {'ground': [741, 777], 'sensor': [741, 777], 'left_speed': 44, 'right_speed': 43}, {'ground': [720, 760], 'sensor': [720, 760], 'left_speed': 45, 'right_speed': 44}, {'ground': [708, 760], 'sensor': [708, 760], 'left_speed': 46, 'right_speed': 45}, {'ground': [712, 752], 'sensor': [712, 752], 'left_speed': 48, 'right_speed': 47}, {'ground': [707, 754], 'sensor': [707, 754], 'left_speed': 49, 'right_speed': 51}, {'ground': [711, 754], 'sensor': [711, 754], 'left_speed': 50, 'right_speed': 53}, {'ground': [711, 769], 'sensor': [711, 769], 'left_speed': 55, 'right_speed': 55}, {'ground': [711, 769], 'sensor': [711, 769], 'left_speed': 55, 'right_speed': 55}, {'ground': [715, 789], 'sensor': [715, 789], 'left_speed': 55, 'right_speed': 50}, {'ground': [712, 794], 'sensor': [712, 794], 'left_speed': 54, 'right_speed': 51}, {'ground': [719, 797], 'sensor': [719, 797], 'left_speed': 54, 'right_speed': 48}, {'ground': [719, 799], 'sensor': [719, 799], 'left_speed': 52, 'right_speed': 48}, {'ground': [732, 792], 'sensor': [732, 792], 'left_speed': 46, 'right_speed': 47}, {'ground': [734, 792], 'sensor': [734, 792], 'left_speed': 44, 'right_speed': 47}, {'ground': [733, 785], 'sensor': [733, 785], 'left_speed': 45, 'right_speed': 48}, {'ground': [730, 781], 'sensor': [730, 781], 'left_speed': 51, 'right_speed': 53}, {'ground': [726, 790], 'sensor': [726, 790], 'left_speed': 56, 'right_speed': 54}, {'ground': [728, 786], 'sensor': [728, 786], 'left_speed': 54, 'right_speed': 53}, {'ground': [723, 786], 'sensor': [723, 786], 'left_speed': 52, 'right_speed': 51}, {'ground': [726, 787], 'sensor': [726, 787], 'left_speed': 50, 'right_speed': 48}]
    lambda_ = 0.39735099337748336

print(f"Our Thymio conversion factor is {lambda_} mm/s per unit")

### Filter selection

During the lectures, we studied four different implementations of Bayes filters:

- `Histogram Filter`: can represent any probability distribution, but it requires discretizing the belief into a finite set of bins, meaning the continuous state space must be approximated by a grid. For real-time applications or high-dimensional state spaces, this approach becomes computationally expensive.

- `Particle Filter`: is conceptually similar to the Histogram Filter because it also discretizes the state space, but instead of using a uniform grid, it maintains a set of samples concentrated around the most probable regions of the belief. It is more efficient than the Histogram Filter, but it remains a non-parametric method, so achieving good accuracy often requires a large number of particles, which increases computational cost.

- `Extended Kalman Filter (EKF)`: generalizes the Kalman Filter to nonlinear systems by linearizing the motion and measurement models. This extension sacrifices optimality, and the filter’s accuracy depends strongly on the quality of the linearization. However, because it is still a parametric filter (representing belief as a Gaussian through a mean and covariance), it is computationally efficient. However, it shares the same limitations as the standard Kalman Filter: it assumes Gaussian noise for both the motion and measurement models, as well as a Gaussian initial belief. 

For our project, we use the `Extended Kalman Filter` because the robot’s motion model is nonlinear, yet the assumptions of Gaussian noise and a Gaussian initial belief remain reasonable. In other words, there is no strong counter-indication that would invalidate these assumptions or justify switching to a non-parametric approach such as the Histogram or Particle Filter. 

Moreover, although the state dimension is small, the computational constraints favor a fast parametric estimator like the EKF rather than a more expensive non-parametric method.

#### Speed Convertion factor

Based on the Exercise 8, we have computed the conversion fator for our Thymio using the following cells which basically compute all the steps of the Exercise 8 required to compute $\lambda$:

### Extended Kalman Filter (EKF)

As described in the [project description document](../docs/project_description.md), the state-space equations are nonlinear due to the projections of the foward and angular velocities of the Thymio. Therefore, we have to use the Extended Kalman Filter to linearized the system at each discrete time k.

Let's consider the previously defined model of the system but with some Gaussian noise incorporated into it:

```math
\begin{equation*}
    \begin{align*}
        \vec{x}_{k} &= 
            \begin{pmatrix}
                x_{k}\\ 
                y_{k}\\
                \theta_{k}
            \end{pmatrix} 
            = \boldsymbol{g}(\vec{x}_{k-1}, \vec{u}_k) = 
            \begin{pmatrix}
                x_{k-1} + dt \cdot \frac{\lambda}{2} (u_{r,k} + u_{l,k}) \cdot cos(\theta_{k-1})\\ 
                y_{k-1} + dt \cdot \frac{\lambda}{2} (u_{r,k} + u_{l,k}) \cdot sin(\theta_{k-1})\\
                \theta_{k-1} - dt \cdot \frac{\lambda}{d} (u_{r,k} - u_{l,k})
            \end{pmatrix} + \epsilon_t \quad \text{with} \quad \vec{u}_k =
                \begin{pmatrix}
                    u_{r,k} \\
                    u_{l,k}
                \end{pmatrix} \quad \text{the odometry.} \\
        \vec{z}_k &= 
            \begin{pmatrix}
                x_{k,measured} \\
                y_{k,measured} \\
                \theta_{k,measured}
            \end{pmatrix} = \boldsymbol{h}(\vec{x}_{k}) =
            \begin{pmatrix}
                x_k \\
                y_k \\
                \theta_k
            \end{pmatrix} + \delta_t
    \end{align*}
\end{equation*}
```

Where $\epsilon_t \sim N(0, Q)$ is a multinormal modelling the uncertainty introduced by the state-space transition, and $\delta_t \sim N(0, R)$ is a multinormal modelling the measurement noise.

As the EKF is an implementation of the general Bayes filter, its algorithm is also decomposed into prediction and update steps. The prediction step is given by the following formula:

```math
\begin{equation*}
    \begin{align*}
        \bar{\mu}_k &= g(\mu_{k-1}, u_k) \\
        \bar{\Sigma}_k &= G_k \Sigma_{k-1} G_k^T + Q_k
    \end{align*}
\end{equation*}
```

And the measurement update step is given by the following:

```math
\begin{equation*}
    \begin{align*}
        K_k &= \bar{\Sigma}_k H_k^T (H_k \bar{\Sigma}_k H_k^T + R_k)^{-1} \\
        \mu_k &= \bar{\mu}_k + K_k (z_k - h(\bar{\mu}_k)) \\
        \Sigma_k &= (I - K_k H_k) \bar{\Sigma}_k
    \end{align*}
\end{equation*}
```

The main difference from the Kalman Filter is that the linear prediction equations are replaced by their nonlinear counterparts, and the linear system matrices are replaced by the corresponding Jacobians.

#### Jacobians matrices

From the previously defined motion model (or state-transition model), we can derive the following Jacobian $G_k$:

```math
\begin{equation}
    \begin{align*}
        G_k &= 
        \begin{pmatrix}
            \frac{\partial g_1}{\partial x_{k-1}} & \frac{\partial g_1}{\partial y_{k-1}} & \frac{\partial g_1}{\partial \theta_{k-1}} \\
            \frac{\partial g_2}{\partial x_{k-1}} & \frac{\partial g_2}{\partial y_{k-1}} & \frac{\partial g_2}{\partial \theta_{k-1}} \\
            \frac{\partial g_3}{\partial x_{k-1}} & \frac{\partial g_3}{\partial y_{k-1}} & \frac{\partial g_3}{\partial \theta_{k-1}}
        \end{pmatrix} \\
        &=
        \begin{pmatrix}
            1 & 0 & -dt \cdot \frac{\lambda}{2} (u_{r,k} + u_{l,k}) \cdot sin(\theta_{k-1}) \\
            0 & 1 & dt \cdot \frac{\lambda}{2} (u_{r,k} + u_{l,k}) \cdot cos(\theta_{k-1}) \\
            0 & 0 & 1
        \end{pmatrix}
    \end{align*}
\end{equation}
```

Similarly, we can derive the following Jacobian, $H_k$, for the measurement model:

```math
\begin{equation}
    \begin{align*}
        H_k &= 
        \begin{pmatrix}
            \frac{\partial h_1}{\partial x_k} & \frac{\partial h_1}{\partial y_k} & \frac{\partial h_1}{\partial \theta_k} \\
            \frac{\partial h_2}{\partial x_k} & \frac{\partial h_2}{\partial y_k} & \frac{\partial h_2}{\partial \theta_k} \\
            \frac{\partial h_3}{\partial x_k} & \frac{\partial h_3}{\partial y_k} & \frac{\partial h_3}{\partial \theta_k}
        \end{pmatrix} \\
        &=
        \begin{pmatrix}
            1 & 0 & 0 \\
            0 & 1 & 0 \\
            0 & 0 & 1
        \end{pmatrix} = I_{3x3}
    \end{align*}
\end{equation}
```

#### File structure

The entire code related to the Extended Kalman Filter can be found in the folder `src/pose_estimation`. 

The file name `nonlinear_system.py` is only an interface that defines the fonctions required in a generic non-linear system. Hence, `differential_drive_system.py` implements it and defines the above system. 

Finally, `extended_kalman_filter.py` receives a non-linear instance and performs the prediction and update step.

#### Noise estimation

Finally, to implement the EKF, we must estimate the noise characteristics $Q_k$ and $R_k$​, which capture the robot’s motion uncertainty and sensor limitations. The more accurately these covariance matrices reflect the real system behavior, the more effective and reliable the EKF becomes.

1. We need to estimate the covariance $Q_k$ of the stochastic noise $\epsilon_t \sim N(0, Q)$ affecting the state-transition model.

```math
\begin{equation}
    Q_k = 
    \begin{pmatrix}
        q_x & 0 & 0 \\
        0 & q_y & 0 \\
        0 & 0 & q_\theta
    \end{pmatrix} = ?
\end{equation}
```

2. We need to estimate the covariance $R_k$ of the stochastic noise $\delta_t \sim N(0, R)$ describing the measurement noise.

```math
\begin{equation}
    R_k =
    \begin{pmatrix}
        r_x & 0 & 0 \\
        0 & r_y & 0 \\
        0 & 0 & r_\theta
    \end{pmatrix} = ?
\end{equation}
```

Similirly to Exercise 8, we have decided to measure on a fixed trajectory the entire system and to estimate how to devide the entire covariance matrix into a measurement covariance and a state-transition matrices.

We used a simple straight path (not perfectly straight but close) and run 10 times our application.

Our setup is the following:
- 1. We measured our mm per pixel coeficient for unit conversion
- 2. We defined a start position and uses our `gui` to measure the position in pixel.
- 3. We defined a stop position as follow

In [ ]:
from src.computer_vision.computer_vision import ComputerVisionCore

IMG_HEIGHT_MM = 920
IMG_HEIGHT_PX = 1080

mm_per_pixel = ComputerVisionCore.compute_mm_per_pixel(point_px_1=0, point_px_2=IMG_HEIGHT_PX, real_dist_mm=IMG_HEIGHT_MM)

print("MM per pixel:", mm_per_pixel)

Then, we used those data to compute the covariance as follow:

In [ ]:
import numpy as np

pose_measures = [[1331.5, 466.1, 0.0], [1333.0, 466.9, 0.0], [1340.2, 463.5, 0.0], [1337.0, 466.0, 0.1], [1337.1, 447.0, 0.1], [1339.1, 452.2, 0.1], [1339.1, 473.0, 0.0], [1336.6, 479.0, -0.1], [1338.3, 455.8, 0.0], [1340.5, 444.5, 0.1]]

cov = np.diag(np.diag(np.cov(np.array(pose_measures).T)))

print("Covariance matrix of pose measures:\n", cov)

As we are trusting more the camera because the markers detection is more robust and consistent that the odometry values, we decided to trust arbitrarly five times more the camera. Hence, we have devided the total covariance matrice into $Q$ and $R$ as follow:

In [ ]:
R = cov / 50  # Measurement noise covariance
Q = R * 49  # Process noise covariance
cov = Q + R  # Total covariance

print("Measurement noise covariance R:\n", R, "\n")
print("State transition noise covariance Q:\n", Q)
print("Total covariance cov:\n", cov)



# 3. Global Navigation and Path Planning

The global navigation module is responsible for computing an optimal, collision-free trajectory from the robot’s current pose to a user-defined goal inside the workspace.  

To do so, our pipeline relies on the following steps:


1. **Obstacle processing**
3. **Visibility Graph Construction**  
4. **Shortest-Path Computation**  

This section details the *theoretical background*, the *choices made*, and the *code required* to execute the global navigation module.


### 3.1 Obstacle Processing

The visibility-graph path planning algorithm we use is, by design, a purely geometric
planner: it only receives **a start point**, **a goal point**, and a list of **polygonal obstacles**.
It has **no notion** of:

- robot size or footprint,  
- safe navigable area,  
- workspace boundaries,  

If applied directly to our raw obstacle polygons, the planner could therefore return
paths that are physically impossible for the robot.

To adapt this generic algorithm to our **real robot operating in a bounded environment**, a
series of preprocessing steps is required. These steps transform the raw vision output
into a consistent geometric world model suitable for robust global planning for the robot.

In this project, the obstacle processing pipeline consists of **three key stages**:

1. **Inflation** of obstacles to account for the robot’s radius (Minkowski sum)  
2. **Merging** of overlapping inflated polygons to avoid fragmented geometry  
3. **Clipping** of inflated obstacles to the allowed navigable workspace  




### 3.1.1 Environment Representation


To guarantee collision-free navigation, the robot centre must remain at least one
robot radius away from every obstacle. Instead of changing the algorithm to consider the robot not as a point but a shape, we decided to 
inflate the obstacles by computing the Minkowski sum between each polygon and
a disk of radius equal to the robot radius.

In practice, this is done by using **Shapely’s buffer operator**:

```python
P.buffer(robot_radius_px, resolution=4, join_style=2)
```

Inflated polygons may contain dozens of vertices. This slows down PyVisgraph, we therefore simplify inflated polygons using:

```python
P_inflated.simplify(15.0, preserve_topology=True)
```

This preserves the geometry while drastically improving planning speed.


### 3.1.2 Merging of Inflated Obstacles

After inflation, obstacles that were previously separate may overlap.  
To avoid artifacts and redundant vertices, we merge all inflated polygons using:

```python
merged = unary_union(inflated)
```




### 3.1.3 Clipping to the allowed area

In our real setup the robot must stay inside a **visually observed area** with
**finite size**. 

We first define the full image support so that no path can go outside the camera image.  
Then, because the robot has a non-zero radius \( r \), its **center** must stay at least
a distance \( r \) away from the image border in order for the whole robot body to remain
inside the observed area. We therefore define a **shrunk navigable workspace**

In code, this workspace is represented as a Shapely box:

```python
border = box(
    robot_radius_px,
    robot_radius_px,
    img_width  - robot_radius_px,
    img_height - robot_radius_px
)

### 3.2 Construction of the Visibility Graph


For global path planning, we use **PyVisgraph**, which builds a graph whose nodes are the vertices of obstacles and whose edges represent **line-of-sight segments**.

For two vertices to be connected, the segment must:

1. Not intersect any obstacle interior,  
2. Lie strictly within the navigable workspace.

PyVisgraph does not know anything about the workspace limits, it may therefore generate edges that lie directly on the border af a clip obstacle. To prevent this, we remove any edge whose segment is not strictly inside the allowed navigation area : 


``` python 


g = vg.VisGraph()
g.build(polygons) 

VG = g.visgraph
for edge in VG.get_edges():
    seg = LineString([(edge.p1.x, edge.p1.y),
                      (edge.p2.x, edge.p2.y)])
    if not seg.within(border):
        edges_to_remove.add(edge)

VG.graph[e.p1].discard(e)
VG.graph[e.p2].discard(e)
VG.edges.discard(e)

## 3.3 Shortest Path Computation

Once the visibility graph has been constructed and all invalid edges removed, we insert the
**start** and **goal** points directly into the graph:  
PyVisgraph automatically connects these points to all visible vertices, and we can query the
optimal path using its built-in Dijkstra implementation.

Since the library raises a `KeyError` when no path exists we catch it:

```python
try:
    path_pts = g.shortest_path(s, t)
except KeyError:
    return None
```

The resulting path is a sequence of straight-line segments forming the globally optimal trajectory.


In [ ]:
img_path = Path.cwd() / "imgs" / "final_path.png"   # ajuste si l’image est ailleurs

img = Image.open(img_path)
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
def plan_path(obstacles, start, goal, robot_radius_px=0, img_width=None, img_height=None, debug_img=None):

        border = box(
            robot_radius_px,
            robot_radius_px,
            img_width - robot_radius_px,
            img_height - robot_radius_px
        )

        # 1) Inflate obstacles
        inflated = GlobalNavigator.inflate_obstacles(obstacles, robot_radius_px)

        # 2) Merge overlapping inflated obstacles
        merged_inflated = GlobalNavigator.merge_inflated_obstacles(inflated)

        # 3) Clip obstacles to the allowed area
        merged_inflated_clip = GlobalNavigator.clip_obstacles_to_image(merged_inflated, border)


        # 4) Validate start and goal positions
        if not GlobalNavigator.outside_obstacle(merged_inflated, start, robot_radius_px/2):
            print("[GLOBAL] Start position invalid:", start)
            return None
        if not GlobalNavigator.outside_obstacle(merged_inflated, goal, robot_radius_px/2):
            print("[GLOBAL] Goal position invalid:", goal)
            return None

        # 5) Compute shortest path using visibility graph
        path, g = GlobalNavigator.compute_visibility_path(merged_inflated_clip, start, goal,border)
        if not GlobalNavigator.path_is_collision_free(path, obstacles) and path != None:
            print("[GLOBAL] No collision-free path found.")
            return None

        # 6) Optional debug drawing
        if debug_img is not None:
            debug_copy = debug_img.copy()
            for poly in obstacles:
                cv2.polylines(debug_copy, [np.array(poly, np.int32)], True, (0, 255, 0), 2)
            for poly in merged_inflated_clip:
                cv2.polylines(debug_copy, [np.array(poly, np.int32)], True, (0, 0, 255), 2)
            for edge in g.visgraph.get_edges():
                cv2.line(debug_copy, (int(edge.p1.x), int(edge.p1.y)),
                         (int(edge.p2.x), int(edge.p2.y)), (255, 255, 0), 1)
            cv2.circle(debug_copy, start, 8, (255, 0, 0), -1)
            cv2.circle(debug_copy, goal, 8, (0, 0, 255), -1)
            for i in range(len(path) - 1):
                cv2.line(debug_copy, tuple(map(int, path[i])), tuple(map(int, path[i + 1])), (255, 0, 0), 3)
            
            return path, debug_copy

        print("[GLOBAL] Path found with", len(path), "points.")
        return path, None

# 4.  Local Navigation

- Our local navigation combines path following and obstacle avoidance. As discussed previously, global navigation plans the global path from robot’s starting point to the end goal taking into account global obstacles.  
- This global path is computed with respect to the **world frame in world coordinates (pixels)**.  
- The local navigation’s purpose is to **allow the robot to avoid obstacles** that are placed at any point along the path by “bending” around them to rejoin the path and reach the goal successfully. These 3D obstacles are to be called **local obstacles**.  

---

**Path Densification**  

- The global path consists of small number of waypoints. Having the distance between two successive waypoints larger than what the robot travels in one period might lead to discontinuous commands and jerky motion. To address this, we densify the polyline before navigation. We insert intermediate points so that the distance between consecutive samples is at most a fixed step we set.  
- This gives us a path with smoothly varying curvature which ensures that the next point the robot is looking for along the path, the lookahead point, is always a nearby waypoint ahead of the robot.  


**Robot-Centered Occupancy Grid**

Robot-Centered Occupancy Grid  
Thymio’s horizontal proximity sensors are used to detect obstacles in its surroundings. We build a local occupancy grid centered on the robot: 

- The grid covers a square region around the robot 
- The grid is centered on the robot’s frame which is defined by:  
  • X axis: positive to the left of the robot  
  • Y axis: positive forward  

Each grid cell stores a scalar value between 0 and 1 representing how likely that cell is occupied with an obstacle. 


In [ ]:
grid = LocalOccupancyGrid(
    size_m=0.8,
    resolution_m=0.01,
    hit_inc=0.4,
    decay=0.85,
    robot_radius_m=0.12,
)



To update the grid, we go through several conversions. To translate what the sensors see in sensor values to the actual distance with respect to the robot’s frame: 

1- Each raw proximity sensor value is converted to distance using a calibration lookup based on mapping between the sensor measurements and sensor distances.  
sensor_val_to_cm_dist(val: int) -> float

2- Obstacle point is estimated in the robot frame (xleft, yforward) by knowing the sensor positions and orientations from the robot’s center. 
obstacles_pos_from_sensor_vals(sensor_vals)

3- This point is mapped to the closest grid cell and the cell’s occupancy value is increased by a fixed increment.  
point_to_index(self, x_m, y_m)

**Inflation:** To be extra careful about obstacles, we inflate the obstacles radius in the grid so that we create a safety margin.  

**Decay:** At every update we multiply the whole grid by a factor less than 1. Old obstacles slowly fade away if they are not seen again. This prevents the robot from remembering obstacles that have moved. 


In [ ]:
def update_from_sensor_vals(self, sensor_vals):
        """
        Update grid from Thymio horizontal proximity readings.
        Obstacles are inflated by robot radius.
        """
        self.grid *= self.decay
        obs_cm = obstacles_pos_from_sensor_vals(sensor_vals)

        inflate_cells = max(1, int(self.robot_radius_m / self.resolution))

        for ox_cm, oy_cm in obs_cm:
            if not np.isfinite(ox_cm) or not np.isfinite(oy_cm):
                continue

            x_m = ox_cm / 100.0
            y_m = oy_cm / 100.0

            idx = self._point_to_index(x_m, y_m)
            if idx is None:
                continue

            iy, ix = idx

            # Inflate in a disk around (iy, ix)
            for dy in range(-inflate_cells, inflate_cells + 1):
                for dx in range(-inflate_cells, inflate_cells + 1):
                    jy = iy + dy
                    jx = ix + dx
                    if 0 <= jy < self.n_cells and 0 <= jx < self.n_cells:
                        if dx * dx + dy * dy <= inflate_cells * inflate_cells:
                            self.grid[jy, jx] = min(
                                1.0, self.grid[jy, jx] + self.hit_inc
                            )

**Lookahead point on the global path**

Lookahead point on the global path  
At each control step, the robot’s pose is found with respect to the global path. From that closest path point, we move forward a desired lookahead distance. The resulting point is the lookahead point.  

This approach is very similar to a pure-pursuit strategy. The robot is always pursuing a point slightly ahead of itself on the path. 


### Artificial Potential field and Virtual Goal

**Artificial Potential field and Virtual Goal**

Following a lookahead point fails when obstacles are placed on the path. Therefore, to make the robot deviate smoothly around obstacles, we define a virtual goal using an artificial potential field in the robot frame. 

We transform the lookahead point from the world frame to robot frame giving a vector pLA = (xleft, yforward). 

1- Attractive component  
To encourage motion towards the lookahead point along the path, we calculate an attractive force which is the lookahead vector in robot frame normalized and scaled by an attraction gain we specify and tune. 
$$
\mathbf{F}_{\text{att}} = k_{\text{att}} \frac{\mathbf{p}_{\text{LA,robot}}}{\left\lVert \mathbf{p}_{\text{LA,robot}} \right\rVert}.
$$


2- Repulsive Component  
To find the repulsive force, we scan all the grid cells whose occupancy is above a threshold. For each cell, we convert the cell index back to an obstacle coordinate (xm, ym) in meters ignoring cells that are behind the robot. Then, a direction that pushes the robot away from the obstacle is calculated, scaled by a repulsive gain Krep and occupancy value. Summing over all cells gives the repulsive force Frep. 

3- Total field and virtual goal  
The total force in robot coordinates is: 
$$
\mathbf{F}_{\text{tot}} = \mathbf{F}_{\text{att}} + \mathbf{F}_{\text{rep}}.
$$
  
If no significant obstacles are seen, Frep is explicitly set to 0 so that the robot follows the global path exactly. Then, that total force is normalized and multiplied by a set distance. The resulting point in the robot frame is the virtual goal. We convert it back to the world coordinates.

The controller uses this virtual goal as its next goal. The potential field takes care of shifting the virtual goal left or right when the grid detects an obstacle on the path. 


In [ ]:
def compute_virtual_goal(self, pose, path, sensor_vals=None):
        if path is None or len(path) == 0:
            return None, None, None, None

        path_arr = np.asarray(path, dtype=float)
        LA_world, _ = self._find_lookahead_point(pose, path_arr)
        if LA_world is None:
            return None, None, None, None

        p_LA_robot = self.world_to_robot(pose, LA_world)

        F_att = p_LA_robot.astype(float)
        norm_att = np.linalg.norm(F_att)
        if norm_att > 1e-6:
            F_att = (self.cfg.k_att / norm_att) * F_att
        print("print attractive force",F_att)
        F_rep = self._repulsive_vector()

        if sensor_vals is None or max(sensor_vals) < 300:
            F_rep = np.zeros(2)

        F_tot = F_att + F_rep

        if sensor_vals is not None:
            left_side  = max(sensor_vals[0], sensor_vals[1])
            right_side = max(sensor_vals[3], sensor_vals[4])

            if left_side > 1200:
                F_tot[0] -= 0.25
            if right_side > 1200:
                F_tot[0] += 0.25

        if sensor_vals is not None and sensor_vals[2] > 1500:
            F_tot[1] = min(F_tot[1], 0.05)

        norm_tot = np.linalg.norm(F_tot)
        if norm_tot < 1e-6:
            F_tot = np.array([0.0, 1.0], dtype=float)
            norm_tot = 1.0

        dir_robot = F_tot / norm_tot
        virt_robot = self.cfg.virt_goal_dist_m * dir_robot
        virt_world = self.robot_to_world(pose, virt_robot)

        return virt_world, LA_world, F_att, F_rep

# 5. Low-Level Controller Overview

Once the local navigator has selected a virtual goal in world coordinates, we use a low-level controller to convert the robot pose and virtual goal into wheel commands for the Thymio.

We model the robot as a unicycle with state \((x, y, theta)\) and control inputs:

- linear velocity v (forward speed)
- angular velocity \(omega) (turning rate)

The controller runs at a fixed period \(dt\) and performs:

1. Computation of the **distance** and **heading error** to the goal.
2. A **PID controller on the heading error** and a proportional law on distance.
3. **Safety heuristics** that react directly to proximity sensors.
4. Mapping of \((v, omega)) to **left/right wheel speeds** and then to motor commands.


In [ ]:
def compute_motor_commands(self, pose, goal_xy, sensor_vals=None,
                               tol=0.02, stop_if_close=True, dt=0.1):
        """
        Same as before, but now:
        - Uses side + center heuristics (if sensor_vals provided).
        - Returns v,w in the info dict (for odometry).
        """
        v, w, rho, alpha = self.compute_unicycle_cmd(pose, goal_xy, dt=dt)
        stopped = False

        if stop_if_close and rho < tol:
            v = 0.0
            w = 0.0
            stopped = True
        else:
            if sensor_vals is not None:
                # side heuristic: choose turn direction + min |w|
                w = apply_side_heuristic(sensor_vals, w,
                                         min_turn=1.5,
                                         side_thr=1200)
                # center heuristic: strong reaction if front-center is high
                v, w = apply_center_heuristic(sensor_vals, v, w,
                                              min_turn=1.8,
                                              center_thr=1500)

            # re-clamp after heuristics
            v = max(-self.g.v_max, min(self.g.v_max, v))
            w = max(-self.g.w_max, min(self.g.w_max, w))

        uL, uR = self.unicycle_to_wheels(v, w)
        return uL, uR, {"rho": rho, "alpha": alpha,
                        "stopped": stopped, "v": v, "w": w}



**Geometry: distance and heading error**

Given the robot pose \((x, y, \theta)\) and the virtual goal position \((x_g, y_g)\) in the world frame, we first compute:

- The vector from robot to goal:
  \[
  \Delta x = x_g - x,\quad \Delta y = y_g - y
  \]

- The **distance** to the goal:
  \[
  \rho = \sqrt{\Delta x^2 + \Delta y^2}
  \]

To reason about “left/right” and “forward/backward”, we express this vector in the **robot frame**. We use the convention:

- \(x_\text{left} > 0\): goal is to the left of the robot  
- \(x_\text{left} < 0\): goal is to the right  
- \(y_\text{forward} > 0\): goal is in front of the robot  

The transformation from world to robot frame is:
\[
\begin{aligned}
y_\text{forward} &= \cos\theta \,\Delta x + \sin\theta \,\Delta y,\\
x_\text{left}    &= \sin\theta \,\Delta x - \cos\theta \,\Delta y.
\end{aligned}
\]

The **heading error** \(\alpha\) is then defined as the angle between the robot’s current heading and the direction to the goal:
\[
\alpha = \operatorname{atan2}(x_\text{left},\, y_\text{forward}).
\]

We wrap \(\alpha\) to \((-\pi, \pi]\) (using `wrap_to_pi`) and apply a small dead-zone of about \(5^\circ\): if \(|\alpha|\) is smaller than this threshold, we set \(\alpha = 0\). This avoids unnecessary oscillations due to tiny angular errors or noise.


**Linear velocity: proportional in distance**

The linear velocity command is chosen as a simple proportional feedback on the distance:

\[
v = K_v \,\rho.
\]

As the robot approaches the virtual goal, \(\rho\) decreases and so does \(v\), which naturally slows the robot down. 

**Angular velocity: PID on heading error**

The angular velocity is more sensitive, because it determines how quickly the robot re-orients towards the goal. A pure proportional law \(\omega = K_\omega \alpha\) gave acceptable behaviour but caused noticeable oscillations around the path.

To improve this, we implemented a **PID controller** on the heading error:

\[
\omega = K_p \alpha
       + K_i \int \alpha\, dt
       + K_d \frac{d\alpha}{dt}.
\]

In the code, we maintain the following state:

- `prev_alpha`: heading error at the previous control step.
- `int_alpha`: accumulated integral \(\int \alpha\, dt\).
- `prev_time`: last update time, used to estimate \(dt\).

At each step:

1. We compute / clamp the timestep $dt$.

2. We update the integral term:  

$$
\texttt{int\_alpha} \leftarrow \texttt{int\_alpha} + \alpha\, dt
$$

and clamp it to $[-\texttt{int\_alpha\_max},\, \texttt{int\_alpha\_max}]$ for **anti-windup**.

3. We compute the derivative:  

$$
\dot{\alpha} = \frac{\alpha - \texttt{prev\_alpha}}{dt}
$$

4. We compute  
$$
\omega = K_p\,\alpha + K_i\,\text{int\_alpha} + K_d\,\dot{\alpha}
$$

and then saturate $\omega$ to $[-\omega_{\max},\,\omega_{\max}]$.
   

The gains \((K_p, K_i, K_d)\) were tuned experimentally on the real robot. We started with a pure P controller, then added a derivative term to damp overshoot, and finally a small integral term to remove steady-state angular errors that appeared due to path curvature and obstacle heuristics.


In [ ]:
def compute_unicycle_cmd(self, pose, goal_xy, dt=None):
        x, y, theta = pose
        xg, yg = goal_xy
        dx = xg - x
        dy = yg - y

        rho = math.hypot(dx, dy)
        print("x:", x,"xg:",xg, "y:", y,"yg:", yg, "dx", dx, "dy", dy, "rho", rho)

        # angle of the goal in robot frame
        y_forward = math.cos(theta) * dx + math.sin(theta) * dy
        x_left = math.sin(theta) * dx - math.cos(theta) * dy
        print("x:", x_left, "y:", y_forward)

        alpha = math.atan2(x_left, y_forward)
        print("alpha", alpha)
        alpha = wrap_to_pi(alpha)
        print("wraped alpha", alpha)

        # deadzone on angle
        alpha_dead = 5.0 * math.pi / 180.0
        if abs(alpha) < alpha_dead:
            alpha = 0.0
        print("dead_alpha", alpha)

        # ---------------------------
        #   PID ON ANGLE (alpha)
        # ---------------------------

        # 1) Determine dt
        if dt is None:
            now = time.time()
            if self.prev_time is None:
                dt = 0.1  # reasonable default first step
            else:
                dt = now - self.prev_time
                # clamp dt so spikes don't explode derivative & integral
                dt = max(0.01, min(dt, 0.2))
            self.prev_time = now
        else:
            # if user passes dt, we just clamp it
            dt = max(0.001, float(dt))

        # 2) Integral term (with anti-windup)
        self.int_alpha += alpha * dt
        # clamp integral to avoid windup
        self.int_alpha = max(-self.g.int_alpha_max,
                             min(self.g.int_alpha_max, self.int_alpha))

        # 3) Derivative term
        alpha_dot = (alpha - self.prev_alpha) / dt
        self.prev_alpha = alpha

        # 2) Integral term (with anti-windup)
        self.int_rho += rho * dt



        # 4) PID output for angular velocity
        Kp = self.g.Komega
        Ki = self.g.Ki_omega
        Kd = self.g.Kd_omega

        w = Kp * alpha + Ki * self.int_alpha + Kd * alpha_dot

        Kiv = self.g.Ki_v

        # 5) Linear velocity 
        v = self.g.Kv * rho + Kiv * self.int_rho

        print(f"PID terms: P={Kp*alpha:.3f}, I={Ki*self.int_alpha:.3f}, "
              f"D={Kd*alpha_dot:.3f}, w_raw={w:.3f}")
        print("v,w before clamp:", v, "  ", w)

        # Clamp
        v = max(-self.g.v_max, min(self.g.v_max, v))
        w = max(-self.g.w_max, min(self.g.w_max, w))
        print("v,w after clamp:", v, "  ", w)

        return v, w, rho, alpha

### Safety heuristics using proximity sensors

On top of the PID controller, we apply simple **heuristics** that react directly to the Thymio’s horizontal proximity sensors:

- Sensors 0–1: front-left side  
- Sensor 2: front-center  
- Sensors 3–4: front-right side  

Two helper functions are used:

1. **Side heuristic** (`apply_side_heuristic`)

   - If the right-hand sensors see a large value (obstacle on the right), we **force a left turn** by ensuring \(\omega \ge \omega_\text{min} > 0\).
   - If the left-hand sensors dominate, we enforce a right turn (\(\omega \le -\omega_\text{min}\)).
   - This only activates when sensor values exceed a threshold (e.g. 1200), so it acts as an “emergency” behaviour when the robot approaches a wall on one side.

2. **Center heuristic** (`apply_center_heuristic`)

   - When the front-center sensor is very high (e.g. > 1500), we reduce the forward velocity \(v\) and enforce a larger minimum \(|\omega|\), choosing the turning direction depending on which side is also seeing the obstacle more.
   - This makes the robot pivot more aggressively away from obstacles directly in front of it and slows down to reduce the risk of collision.

These heuristics are applied after the PID computation and before the final saturation. They do not replace the controller, but bias \(\omega\) and slightly reduce \(v\) when collision risk is high.


In [ ]:
def apply_side_heuristic(sensor_vals, w, min_turn=1.5, side_thr=1200):
    """
    Use outer front sensors to enforce TURN DIRECTION and minimum |w|.

    Thymio prox.horizontal:
        0: front left
        1: front left-center
        2: front center
        3: front right-center
        4: front right
        5: rear left
        6: rear right

    - If right side is stronger -> force LEFT turn (w > 0)
    - If left  side is stronger -> force RIGHT turn (w < 0)
    """
    if sensor_vals is None or len(sensor_vals) < 5:
        return w

    s = sensor_vals
    left_side = max(s[0], s[1])
    right_side = max(s[3], s[4])

    if left_side < side_thr and right_side < side_thr:
        return w

    if right_side > left_side and right_side >= side_thr:
        # obstacle on RIGHT -> turn LEFT
        if w <= 0:
            w = min_turn
        else:
            w = max(w, min_turn)
    elif left_side > right_side and left_side >= side_thr:
        # obstacle on LEFT -> turn RIGHT
        if w >= 0:
            w = -min_turn
        else:
            w = min(w, -min_turn)

    return w


def apply_center_heuristic(sensor_vals, v, w, min_turn=1.8, center_thr=1500):
    """
    Special handling when the FRONT-CENTER sensor (index 2) sees something.

    - Strong turn left/right depending on which side also sees the obstacle more.
    - Slightly reduce v so the robot turns harder and doesn't hit the obstacle.
    """
    if sensor_vals is None or len(sensor_vals) < 3:
        return v, w

    s = sensor_vals
    center_val = s[2]

    if center_val < center_thr:
        return v, w

    left_side = max(s[0], s[1])
    right_side = max(s[3], s[4])

    if left_side < center_thr and right_side < center_thr:
        # Both sides small -> just enforce min |w| with current sign
        if w >= 0:
            w = max(w, min_turn)
        else:
            w = min(w, -min_turn)
    else:
        # Stronger on the right -> turn left
        if right_side >= left_side:
            if w <= 0:
                w = min_turn
            else:
                w = max(w, min_turn)
        # Stronger on the left -> turn right
        else:
            if w >= 0:
                w = -min_turn
            else:
                w = min(w, -min_turn)

    # reduce forward speed when something is dead ahead
    v = min(v, 0.05)
    return v, w